# EERIE IFS-FESOM native-grid North Sea extraction

This executable notebook retrieves real EERIE IFS-FESOM2-SR atmospheric data from the public DKRZ km-scale cloud. The native atmospheric grid is approximately 9 km and is stored as a reduced Gaussian grid with a one-dimensional `value` axis.

The default demonstrator retrieves the first daily data chunk covering the requested historical period. The cloud stores native variables in chunks spanning all 6,599,680 grid points, so retrieving the complete 1995-2014 native-grid record would require a very large transfer. The notebook deliberately reports the retrieved coverage and does not label the one-day result as a 20-year climatology.

The available native daily variables include `m10u`, `m10v`, `mean10ws`, `mean2t`, and `msp`. This dataset provides 10 m wind, not 100 m wind; hub-height extrapolation is therefore not performed here.

In [1]:
from pathlib import Path
import json
import time
import requests
import numpy as np
import pandas as pd
from numcodecs import Blosc

DATA_DIR = Path('../data').resolve()
DATA_DIR.mkdir(exist_ok=True)
BASE = 'https://km-scale-cloud.dkrz.de/datasets/ifs-fesom2-sr.hist-1950.v20240304.atmos.native.2D_daily_avg/zarr'
DATASET_ID = 'ifs-fesom2-sr.hist-1950.v20240304.atmos.native.2D_daily_avg'
LON_MIN, LON_MAX = 0.0, 10.0
LAT_MIN, LAT_MAX = 50.0, 62.0
REQUESTED_START, REQUESTED_END = '1995-01-01', '2014-12-31'
print('Output directory:', DATA_DIR)
print('Source:', BASE)

Output directory: C:\Users\LENOVO\wind-energy-agentic-analysis\data
Source: https://km-scale-cloud.dkrz.de/datasets/ifs-fesom2-sr.hist-1950.v20240304.atmos.native.2D_daily_avg/zarr


In [2]:
# Read consolidated metadata and verify the dataset layout.
meta = requests.get(BASE + '/.zmetadata', timeout=180).json()['metadata']
print('Native points:', meta['lat/.zarray']['shape'][0])
print('Coordinate chunk shape:', meta['lat/.zarray']['chunks'])
print('Variable chunk shape:', meta['m10u/.zarray']['chunks'])

def get_chunk(key):
    response = requests.get(f'{BASE}/{key}', timeout=600)
    response.raise_for_status()
    return response.content

def decode_chunk(key, dtype, shape):
    raw = get_chunk(key)
    decoded = Blosc().decode(raw)
    return np.frombuffer(decoded, dtype=np.dtype(dtype)).reshape(shape)

# Coordinates are one compressed chunk each.
lat = decode_chunk('lat/0', '<f8', (6599680,))
lon = decode_chunk('lon/0', '<f8', (6599680,))
mask = (lon >= LON_MIN) & (lon <= LON_MAX) & (lat >= LAT_MIN) & (lat <= LAT_MAX)
indices = np.flatnonzero(mask)
print('North Sea native points:', len(indices))
print('Latitude range:', float(lat[indices].min()), float(lat[indices].max()))
print('Longitude range:', float(lon[indices].min()), float(lon[indices].max()))

Native points: 6599680
Coordinate chunk shape: [6599680]
Variable chunk shape: [2, 6599680]


North Sea native points: 9367
Latitude range: 50.01757338975612 61.96836350182526
Longitude range: 0.0 10.0


In [3]:
# Variable chunks contain two daily records. Select the chunk containing the requested start date.
time_meta = meta['time/.zarray']
variable_chunk_days = int(meta['m10u/.zarray']['chunks'][0])
DATASET_START = pd.Timestamp('1950-01-01')
target_start = pd.Timestamp(REQUESTED_START)
DATA_CHUNK_INDEX = int((target_start - DATASET_START).days // variable_chunk_days)
RETRIEVED_TIMES = pd.date_range(DATASET_START + pd.Timedelta(days=DATA_CHUNK_INDEX * variable_chunk_days), periods=variable_chunk_days, freq='D')
print('Variable chunk size:', variable_chunk_days, 'days')
print('Selected chunk:', DATA_CHUNK_INDEX)
print('Retrieved dates:', RETRIEVED_TIMES[0], 'to', RETRIEVED_TIMES[-1])

Variable chunk size: 2 days
Selected chunk: 8218
Retrieved dates: 1995-01-01 00:00:00 to 1995-01-02 00:00:00


In [4]:
# Retrieve only the North Sea points from the two-day chunks for wind, temperature, and pressure.
# Each source chunk is large because it spans the complete native grid; after decoding, we discard all other points.
variable_specs = {
    'm10u': ('<f4', (2, 6599680)),
    'm10v': ('<f4', (2, 6599680)),
    'mean2t': ('<f4', (2, 6599680)),
    'msp': ('<f4', (2, 6599680)),
}
regional = {}
for name, (dtype, shape) in variable_specs.items():
    started = time.time()
    values = decode_chunk(f'{name}/{DATA_CHUNK_INDEX}.0', dtype, shape)
    regional[name] = values[:, indices]
    print(name, regional[name].shape, 'retrieved in', round(time.time() - started, 1), 's')

u = regional['m10u']
v = regional['m10v']
temperature_k = regional['mean2t']
pressure_pa = regional['msp']
wind_speed = np.hypot(u, v)
air_density = pressure_pa / (287.05 * temperature_k)
wpd = 0.5 * air_density * wind_speed ** 3
power_ws = np.array([0,3,4,5,6,7,8,9,10,11,12,13,14,15,20,25,30], dtype=float)
power_mw = np.array([0,0,.5,1.5,3,5,7,9,11,12.5,14,14.8,15,15,15,15,0], dtype=float)
capacity_factor = np.interp(wind_speed, power_ws, power_mw, left=0, right=0) / 15.0
print('Mean 10 m wind speed:', float(np.nanmean(wind_speed)))
print('Mean WPD:', float(np.nanmean(wpd)), 'W/m2')
print('Illustrative 15 MW CF:', float(np.nanmean(capacity_factor)))

m10u (2, 9367) retrieved in 65.4 s


m10v (2, 9367) retrieved in 73.9 s


mean2t (2, 9367) retrieved in 43.5 s


msp (2, 9367) retrieved in 31.7 s
Mean 10 m wind speed: 8.605984687805176
Mean WPD: 632.8680419921875 W/m2
Illustrative 15 MW CF: 0.5100502219131751


In [5]:
# Save actual high-resolution sample outputs.
rows = []
for t in range(wind_speed.shape[0]):
    rows.append(pd.DataFrame({
        'time': RETRIEVED_TIMES[t],
        'lat': lat[indices],
        'lon': lon[indices],
        'wind_speed_10m_ms': wind_speed[t],
        'air_density_kgm3': air_density[t],
        'wind_power_density_wm2': wpd[t],
        'capacity_factor_15mw_illustrative': capacity_factor[t],
    }))
sample = pd.concat(rows, ignore_index=True)
sample.to_parquet(DATA_DIR / 'north_sea_highres_sample_points.parquet', index=False)
sample.to_csv(DATA_DIR / 'north_sea_highres_sample_points.csv', index=False)

provenance = {
    'dataset_id': DATASET_ID,
    'source_url': BASE,
    'requested_period': [REQUESTED_START, REQUESTED_END],
    'retrieved_period': [str(RETRIEVED_TIMES[0].date()), str(RETRIEVED_TIMES[-1].date())],
    'retrieved_timesteps': int(wind_speed.shape[0]),
    'native_atmospheric_resolution': 'approximately 9 km',
    'native_grid': 'reduced Gaussian, 6599680 points',
    'north_sea_points': int(len(indices)),
    'variables': ['m10u', 'm10v', 'mean2t', 'msp'],
    'wind_reference_height': '10 m',
    'limitation': 'A complete 1995-2014 native-grid extraction requires a very large transfer because source chunks span all native points; this output is a connectivity demonstrator, not a climatology.',
}
with open(DATA_DIR / 'highres_sample_provenance.json', 'w', encoding='utf-8') as f:
    json.dump(provenance, f, indent=2)
print('Saved:', DATA_DIR / 'north_sea_highres_sample_points.parquet')
print('Saved:', DATA_DIR / 'north_sea_highres_sample_points.csv')
print('Saved:', DATA_DIR / 'highres_sample_provenance.json')

Saved: C:\Users\LENOVO\wind-energy-agentic-analysis\data\north_sea_highres_sample_points.parquet
Saved: C:\Users\LENOVO\wind-energy-agentic-analysis\data\north_sea_highres_sample_points.csv
Saved: C:\Users\LENOVO\wind-energy-agentic-analysis\data\highres_sample_provenance.json
